In [1]:
import sys
sys.path.append("..")

from source.system import SystemParams
from source.policy import NeuralPolicy

from source.utils import read_csv
from source.training import train_net

In [8]:
import torch
import os

EPOCHS       = 1000
LR           = 5e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE   = 1000
SEED_BASE    = 123
LAMBDA_COST  = [0.0, 0.1, 0.25, 0.5]
ALPHA_GRID   = 0.0
DEVICE       = torch.device("cpu")

params = SystemParams()
S0_single = torch.tensor([[0., 0., 100., 0.01]], device=DEVICE)

energy_prices = read_csv("../data/eprice_test.csv")
price_history = energy_prices[:params.T]
future_price = energy_prices[params.T:]

CHECKPOINT_DIR = "../checkpoints/lambda_experiments"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Same initial weights for all lambda experiments
torch.manual_seed(SEED_BASE)
initial_policy = NeuralPolicy().to(DEVICE)
initial_state = {
    k: v.detach().cpu().clone()
    for k, v in initial_policy.state_dict().items()
}

for lambda_cost in LAMBDA_COST:

    print("=" * 50)
    print(f"Training: lambda={lambda_cost}")

    # Reset policy to the same initial weights
    policy = NeuralPolicy().to(DEVICE)
    policy.load_state_dict(initial_state)

    opt = torch.optim.AdamW(
        policy.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    res = train_net(
        policy=policy,
        optimizer=opt,
        device=DEVICE,
        params=params,
        S0_single=S0_single,
        endog_history=price_history,
        future_endog=future_price,
        epochs=EPOCHS,
        lambda_cost=lambda_cost,
        alpha_grid=ALPHA_GRID,
        batch_size=BATCH_SIZE,
        seed_base=SEED_BASE,
        verbose=True
    )

    print(f"lambda: {lambda_cost}, best J: {res['best_J']}")

    filename = f"policy_lambda_{str(lambda_cost).replace('.', '_')}.pt"
    path = os.path.join(CHECKPOINT_DIR, filename)

    checkpoint = {
        "lambda_cost": lambda_cost,
        "alpha_grid": ALPHA_GRID,
        "best_J": res["best_J"],
        "policy_state_dict": res["best_state"],
        "J_train_hist": res["J_train_hist"],
        "J_eval_hist": res["J_eval_hist"],
        "seed_base": SEED_BASE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
    }

    if "best_epoch" in res:
        checkpoint["best_epoch"] = res["best_epoch"]

    torch.save(checkpoint, path)

    print(f"Saved best policy: {path}\n")

Training: lambda=0.0
[050] J_train=-9.4682 | J_eval(det)=-9.7222
[100] J_train=-12.9618 | J_eval(det)=-12.9657
[150] J_train=-14.4694 | J_eval(det)=-14.2639
[200] J_train=-17.3172 | J_eval(det)=-16.7000
[250] J_train=-19.1732 | J_eval(det)=-17.9740
[300] J_train=-20.7378 | J_eval(det)=-18.8588
[350] J_train=-21.8021 | J_eval(det)=-19.5487
[400] J_train=-23.1731 | J_eval(det)=-20.1940
[450] J_train=-24.1983 | J_eval(det)=-20.9799
[500] J_train=-25.3548 | J_eval(det)=-21.7083
[550] J_train=-25.8399 | J_eval(det)=-22.1439
[600] J_train=-26.0871 | J_eval(det)=-22.2686
[650] J_train=-26.6671 | J_eval(det)=-22.2942
[700] J_train=-26.9475 | J_eval(det)=-22.4957
[750] J_train=-26.9691 | J_eval(det)=-22.5961
[800] J_train=-26.8747 | J_eval(det)=-22.7431
[850] J_train=-27.0653 | J_eval(det)=-22.7422
[900] J_train=-27.4254 | J_eval(det)=-22.7819
[950] J_train=-27.2867 | J_eval(det)=-22.8708
[1000] J_train=-27.5232 | J_eval(det)=-22.8744

Training ready in 485.2s. Epoch: 996. Best J=-27.9179
lambd